In [ ]:
!pip install -q -U transformers peft trl datasets accelerate bitsandbytes

In [ ]:
import torch, transformers, peft, trl, bitsandbytes as bnb
from transformers.utils import is_bitsandbytes_available
print(transformers.__version__, peft.__version__, trl.__version__)
print(bnb.__version__, is_bitsandbytes_available(), torch.cuda.is_available())

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(UserSecretsClient().get_secret("HF_TOKEN"))

In [2]:
import re, json, torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# data: 2,000 rows -> 1,800 train / 200 test
ds = load_dataset("b-mc2/sql-create-context", split="train")
ds = ds.shuffle(seed=42).select(range(2000))
split = ds.train_test_split(test_size=0.1, seed=42)
train_ds, test_ds = split["train"], split["test"]

SYSTEM = "You are an expert SQL writer. Given a table schema and a question, reply with only the SQL query."

def make_messages(ex, with_answer=False):
    msgs = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": f"Schema:\n{ex['context']}\n\nQuestion: {ex['question']}"},
    ]
    if with_answer:
        msgs.append({"role": "assistant", "content": ex["answer"]})
    return msgs

def normalize(sql):
    sql = sql.strip().strip("`")
    sql = re.sub(r"^sql\s*\n", "", sql, flags=re.I)
    sql = sql.replace('"', "'").strip().rstrip(";")
    sql = re.sub(r"\s+", " ", sql)
    return sql.lower()

model_id = "Qwen/Qwen2.5-1.5B-Instruct"
bnb_cfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_compute_dtype=torch.float16)
tok = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id, quantization_config=bnb_cfg, device_map={"": 0},
    dtype=torch.float16,
)

@torch.no_grad()
def evaluate(model, dataset, batch_size=16, max_new_tokens=128):
    tok.padding_side = "left"
    model.eval()
    correct, preds = 0, []
    for i in range(0, len(dataset), batch_size):
        batch = dataset.select(range(i, min(i + batch_size, len(dataset))))
        prompts = [tok.apply_chat_template(make_messages(ex), tokenize=False,
                   add_generation_prompt=True) for ex in batch]
        inputs = tok(prompts, return_tensors="pt", padding=True).to(model.device)
        out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                             do_sample=False, pad_token_id=tok.eos_token_id)
        texts = tok.batch_decode(out[:, inputs["input_ids"].shape[1]:],
                                 skip_special_tokens=True)
        for ex, pred in zip(batch, texts):
            preds.append(pred)
            correct += normalize(pred) == normalize(ex["answer"])
    return correct / len(dataset), preds

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [3]:
def to_text(ex):
    return {"text": tok.apply_chat_template(make_messages(ex, with_answer=True), tokenize=False)}

train_text = train_ds.map(to_text, remove_columns=train_ds.column_names)
print(train_text[0]["text"])

<|im_start|>system
You are an expert SQL writer. Given a table schema and a question, reply with only the SQL query.<|im_end|>
<|im_start|>user
Schema:
CREATE TABLE table_name_29 (position VARCHAR, notes VARCHAR, year VARCHAR)

Question: What position had notes of 20km and a year earlier than 2002?<|im_end|>
<|im_start|>assistant
SELECT position FROM table_name_29 WHERE notes = "20km" AND year < 2002<|im_end|>



In [5]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

tok.padding_side = "right"

lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,
                  target_modules="all-linear", task_type="CAUSAL_LM")

args = SFTConfig(
    output_dir="/kaggle/working/out",
    num_train_epochs=2,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=10,
    logging_steps=10,
    save_strategy="no",
    fp16=True,
    bf16=False,
    max_length=512,
    dataset_text_field="text",
    report_to="none",
)

trainer = SFTTrainer(model=model, train_dataset=train_text,
                     peft_config=lora, args=args, processing_class=tok)

# keep the trainable LoRA weights in float32 so the fp16 scaler works
for p in trainer.model.parameters():
    if p.requires_grad:
        p.data = p.data.float()

trainer.train()

/usr/local/lib/python3.13/dist-packages/peft/mapping_func.py:148: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/peft/tuners/tuners_utils.py:331: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Step,Training Loss
10,1.996326
20,0.798294
30,0.689101
40,0.688572
50,0.653527
60,0.659803
70,0.649699
80,0.696933
90,0.665783
100,0.643156


TrainOutput(global_step=226, training_loss=0.6902304961618069, metrics={'train_runtime': 508.1622, 'train_samples_per_second': 7.084, 'train_steps_per_second': 0.445, 'total_flos': 3510055346368512.0, 'train_loss': 0.6902304961618069, 'entropy': 0.5708764439279382, 'mean_token_accuracy': 0.867057515816255, 'num_tokens': 370556.0, 'epoch': 2.0})

In [6]:
ft_acc, ft_preds = evaluate(trainer.model, test_ds)
print(f"BEFORE: {base_acc:.1%}   AFTER: {ft_acc:.1%}")

import json
with open("/kaggle/working/results.json", "w") as f:
    json.dump({"base_acc": base_acc, "ft_acc": ft_acc, "n_test": len(test_ds),
               "train_rows": len(train_ds), "epochs": 2, "lora_r": 16, "lr": 2e-4}, f)

for ex, p in list(zip(test_ds, ft_preds))[:5]:
    print("Q:", ex["question"]); print("REF:", ex["answer"]); print("PRED:", p); print("---")

NameError: name 'base_acc' is not defined

In [7]:
with trainer.model.disable_adapter():
    base_acc, base_preds = evaluate(trainer.model, test_ds)

print(f"BEFORE: {base_acc:.1%}   AFTER: {ft_acc:.1%}")

import json
with open("/kaggle/working/results.json", "w") as f:
    json.dump({"base_acc": base_acc, "ft_acc": ft_acc, "n_test": len(test_ds),
               "train_rows": len(train_ds), "epochs": 2, "lora_r": 16, "lr": 2e-4}, f)

for ex, p in list(zip(test_ds, base_preds))[:3]:
    print("BASE | Q:", ex["question"]); print("REF:", ex["answer"]); print("PRED:", p); print("---")
for ex, p in list(zip(test_ds, ft_preds))[:3]:
    print("FT | Q:", ex["question"]); print("REF:", ex["answer"]); print("PRED:", p); print("---")

BEFORE: 46.0%   AFTER: 68.0%
BASE | Q: How many Goals against have Played more than 34?
REF: SELECT COUNT(goals_against) FROM table_name_97 WHERE played > 34
PRED: ```sql
SELECT COUNT(goals_against) FROM table_name_97 WHERE played > 34;
```
---
BASE | Q: What are the names listed in the family  il-1f2?
REF: SELECT name FROM table_29871617_1 WHERE family_name = "IL-1F2"
PRED: ```sql
SELECT name FROM table_29871617_1 WHERE family_name = 'il-1f2';
```
---
BASE | Q: What was Min Patel's bowling?
REF: SELECT bowling FROM table_name_7 WHERE player = "min patel"
PRED: SELECT bowling FROM table_name_7 WHERE player = 'Min Patel'
---
FT | Q: How many Goals against have Played more than 34?
REF: SELECT COUNT(goals_against) FROM table_name_97 WHERE played > 34
PRED: SELECT COUNT(goals_against) FROM table_name_97 WHERE played > 34
---
FT | Q: What are the names listed in the family  il-1f2?
REF: SELECT name FROM table_29871617_1 WHERE family_name = "IL-1F2"
PRED: SELECT name FROM table_29871617_1 W

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch
print(torch.cuda.device_count(), torch.cuda.is_available())

1 True


In [8]:
wrong = [(ex, p) for ex, p in zip(test_ds, ft_preds) if normalize(p) != normalize(ex["answer"])]
print(len(wrong), "wrong out of", len(test_ds))
for ex, p in wrong[:10]:
    print("Q:", ex["question"]); print("REF:", ex["answer"]); print("PRED:", p); print("---")

64 wrong out of 200
Q: How many were in attendance with a loss of Prokopec (2-7)?
REF: SELECT attendance FROM table_name_21 WHERE loss = "prokopec (2-7)"
PRED: SELECT COUNT(attendance) FROM table_name_21 WHERE loss = "prokopec (2-7)"
---
Q: Who was the Rookie of the Year for the Season in 2000?
REF: SELECT rookie_of_the_year FROM table_name_72 WHERE season = 2000
PRED: SELECT rookie_of_the_year FROM table_name_72 WHERE season = "2000"
---
Q: What is the extra result of the 4th game?
REF: SELECT extra FROM table_name_37 WHERE result = "4th"
PRED: SELECT extra FROM table_name_37 WHERE result = "4th game"
---
Q: Count the Number of electorates (2009) that has a District of fatehpur and  jahanabad?
REF: SELECT SUM(number_of_electorates__2009_) FROM table_name_11 WHERE district = "fatehpur" AND name = "jahanabad"
PRED: SELECT AVG(number_of_electorates__2009_) FROM table_name_11 WHERE district = "fatehpur" AND name = "jahanabad"
---
Q: List the details of the customers who do not have any po

In [9]:
trainer.model.save_pretrained("/kaggle/working/text2sql-adapter")
tok.save_pretrained("/kaggle/working/text2sql-adapter")

('/kaggle/working/text2sql-adapter/tokenizer_config.json',
 '/kaggle/working/text2sql-adapter/chat_template.jinja',
 '/kaggle/working/text2sql-adapter/tokenizer.json')

In [11]:
repo = "PratikshaMahajan/qwen2.5-1.5b-text2sql-qlora"
trainer.model.push_to_hub(repo)
tok.push_to_hub(repo)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

CommitInfo(commit_url='https://huggingface.co/PratikshaMahajan/qwen2.5-1.5b-text2sql-qlora/commit/130d35f22d91b8c52f8391cd3f2ff71ce2930fbe', commit_message='Upload tokenizer', commit_description='', oid='130d35f22d91b8c52f8391cd3f2ff71ce2930fbe', pr_url=None, repo_url=RepoUrl('https://huggingface.co/PratikshaMahajan/qwen2.5-1.5b-text2sql-qlora', endpoint='https://huggingface.co', repo_type='model', repo_id='PratikshaMahajan/qwen2.5-1.5b-text2sql-qlora'), pr_revision=None, pr_num=None)